<a href="https://colab.research.google.com/github/hernan-av/proyecto-cloud-analytics/blob/main/notebooks/03_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cloud Provider Analytics · Pipeline

Minería de Datos II · ISTEA · 2C 2026 · Alberto Hernán Avalos y Pedro Alves

Segunda entrega: pipeline ejecutable Landing → Bronze → Silver → Gold → Cassandra (AstraDB).

| Sección | Qué hace | ¿Es parte del pipeline? |
|---|---|---|
| 0. Preparación | Entorno, rutas, registro de ejecución | Soporte |
| 1. Fuente simulada | Hace de sistema de origen: entrega los archivos a Landing | **No**, solo para el proyecto |
| 2. Landing | Lo que llegó, sin modificar, y el registro de llegadas | Sí |
| 3. Bronze | Maestros (batch) y eventos (streaming) a Parquet tipado | Sí |
| 4. Silver | Limpieza, reglas de calidad, cuarentena, unión con maestros | Sí |
| 5. Gold | `org_daily_usage_by_service` | Sí |
| 6. Serving | Carga en Cassandra y consultas | Sí |
| 7. Pruebas e idempotencia | Reejecución sin duplicados y controles | Control |
| 8. Evidencias | Registro, muestras y salidas a `evidence/` | Control |

Las decisiones se anotan en el paso donde se aplican (bloques **Decisión**) y están resumidas en [`DECISIONS.md`](../DECISIONS.md).

> Para correrlo: Entorno de ejecución → Ejecutar todo. La sección 6 necesita el token de AstraDB en los Secretos de Colab (`ASTRA_TOKEN`) y el Secure Connect Bundle; ver el README.

# 0. Preparación

En Colab trabajamos en modo local: una sola máquina. Spark igual arma el plan, lo divide en tareas y las reparte entre los núcleos disponibles, con la misma lógica que en un cluster...

In [136]:
# COLAB: clona o actualiza el repositorio
import os
REPO_URL = "https://github.com/hernan-av/proyecto-cloud-analytics.git"
REPO_DIR = "/content/proyecto-cloud-analytics"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

!pip -q install pyspark cassandra-driver
print("Repo listo en", REPO_DIR)

Repo listo en /content/proyecto-cloud-analytics


## 0.2 Rutas

Hay dos lugares distintos y conviene no mezclarlos:

- **Datos provistos** (`datalake/landing/` del repo): el dataset que entregó la cursada. Son los lotes que se obtienen de los sistemas de origen. No se modifica.
- **Data Lake** (`/content/lake/`): lo que arma el pipeline. Vive en Colab y se regenera en cada corrida. No se sube a GitHub.

In [137]:
import shutil
from pathlib import Path

# Dataset de la cursada (solo lectura)
DATOS_PROVISTOS = Path(REPO_DIR) / "datalake" / "landing"

# Zonas del Data Lake
LAKE        = Path("/content/lake")
LANDING     = LAKE / "landing"
BRONZE      = LAKE / "bronze"
SILVER      = LAKE / "silver"
QUARANTINE  = SILVER / "quarantine"
GOLD        = LAKE / "gold"
CHECKPOINTS = LAKE / "_checkpoints"     # memoria de los procesos de streaming

# Evidencias, se suben al repo
EVIDENCE    = Path(REPO_DIR) / "evidence"

# Corrida limpia: se borra el lake anterior. Los datos originales no se tocan.
if LAKE.exists():
    shutil.rmtree(LAKE)

for p in [LANDING, BRONZE, SILVER, QUARANTINE, GOLD, CHECKPOINTS, EVIDENCE]:
    p.mkdir(parents=True, exist_ok=True)

print("Datos provistos:", DATOS_PROVISTOS)
print("Data Lake:      ", LAKE)

Datos provistos: /content/proyecto-cloud-analytics/datalake/landing
Data Lake:       /content/lake


## 0.3 Spark

In [138]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .master("local[*]")                              # modo local, todos los núcleos de la máquina
    .appName("CloudProviderAnalytics")
    .config("spark.sql.shuffle.partitions", "4")     # pocas particiones: el dataset es chico
    .config("spark.sql.session.timeZone", "UTC")     # las fechas de los datos están en UTC (D11)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Núcleos en uso (paralelismo):", spark.sparkContext.defaultParallelism)
print("Particiones de shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark: 4.0.5
Master: local[*]
Núcleos en uso (paralelismo): 2
Particiones de shuffle: 4


## 0.4 Registro de ejecución

Cada paso anota qué leyó, qué escribió y dónde (observabilidad y linaje). Al final de cada zona se muestra el registro y en la sección 8 se guarda en `evidence/`. (observabilidad y linaje).

In [139]:
import pandas as pd
from datetime import datetime, timedelta

registro = []

def tamano_kb(ruta):
    # Tamaño de los Parquet de una carpeta, en KB
    return round(sum(f.stat().st_size for f in Path(ruta).rglob("*.parquet")) / 1024, 1)

def anotar(etapa, tabla, **datos):
    # Agrega una fila al registro de ejecución
    registro.append({"momento": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
                     "etapa": etapa, "tabla": tabla, **datos})

def ver_registro(etapa=None):
    # Muestra el registro (completo o de una etapa) como tabla
    df = pd.DataFrame(registro)
    if etapa is not None:
        df = df[df["etapa"].str.startswith(etapa)].dropna(axis=1, how="all")
    return df.fillna("")

# 1. Fuente simulada (fuera del pipeline)

Esta sección **no es parte del pipeline**. Hace el papel de los sistemas de origen del diagrama (CRM, facturación, medición de uso…), que en un entorno real mandarían sus archivos a Landing.

Hace falta porque el dataset de la cursada son un "volcado": los 120 archivos de eventos no tienen fecha de llegada y los eventos están mezclados al azar.

## 1.1 Orden de los eventos orginales, como llegan

In [140]:
# medimos con el id y la fecha de cada evento
provistos = (spark.read
             .schema("event_id STRING, timestamp TIMESTAMP")
             .json(str(DATOS_PROVISTOS / "usage_events_stream"))
             .withColumn("archivo", F.col("_metadata.file_name")))

(provistos.groupBy("archivo")
 .agg(F.min("timestamp").alias("evento_mas_viejo"),
      F.max("timestamp").alias("evento_mas_nuevo"),
      F.count("*").alias("eventos"))
 .orderBy("archivo")
 .show(5, truncate=False))

+----------------------+-------------------+-------------------+-------+
|archivo               |evento_mas_viejo   |evento_mas_nuevo   |eventos|
+----------------------+-------------------+-------------------+-------+
|events_part_0000.jsonl|2025-07-03 00:08:00|2025-08-31 16:32:00|360    |
|events_part_0001.jsonl|2025-07-03 02:38:00|2025-08-31 23:51:00|360    |
|events_part_0002.jsonl|2025-07-03 14:16:00|2025-08-31 20:47:00|360    |
|events_part_0003.jsonl|2025-07-03 00:48:00|2025-08-31 23:51:00|360    |
|events_part_0004.jsonl|2025-07-03 10:43:00|2025-08-31 20:40:00|360    |
+----------------------+-------------------+-------------------+-------+
only showing top 5 rows


Cada archivo trae eventos de todo el período (julio y agosto).

## 1.2 Cómo afectan al watermark si llegaran así?

Suponemos que llegan en el orden del nombre, de a 10 archivos por micro-batch, y contamos cuántos eventos descartaría el watermark con cada tolerancia.

In [141]:
# Número de tanda según el número del archivo (10 archivos por tanda)
tandas = provistos.withColumn(
    "tanda", (F.regexp_extract("archivo", r"(\d+)", 1).cast("int") / 10).cast("int"))

# Para cada tanda: el evento más nuevo visto en las tandas anteriores
anteriores = Window.orderBy("tanda").rowsBetween(Window.unboundedPreceding, -1)
max_previo = (tandas.groupBy("tanda").agg(F.max("timestamp").alias("max_ts"))
              .withColumn("max_visto_antes", F.max("max_ts").over(anteriores)))

# Horas que cada evento queda por detrás de lo ya visto
desorden = (tandas.join(max_previo.select("tanda", "max_visto_antes"), "tanda")
            .withColumn("horas", (F.col("max_visto_antes").cast("long") - F.col("timestamp").cast("long")) / 3600)
            .cache())
total = desorden.count()

tolerancias = [("1 hora", 1), ("1 día", 24), ("7 días", 168), ("30 días", 720), ("59 días", 59*24), ("60 días", 60*24)]
filas = [(nombre, desorden.filter(F.col("horas") > h).count()) for nombre, h in tolerancias]

spark.createDataFrame(filas, ["tolerancia", "eventos_descartados"]) \
     .withColumn("pct", F.round(100 * F.col("eventos_descartados") / total, 1)).show()
desorden.unpersist();

+----------+-------------------+----+
|tolerancia|eventos_descartados| pct|
+----------+-------------------+----+
|    1 hora|              39566|91.6|
|     1 día|              38935|90.1|
|    7 días|              34983|81.0|
|   30 días|              19758|45.7|
|   59 días|                657| 1.5|
|   60 días|                  0| 0.0|
+----------+-------------------+----+



> **Decisión D15 · Simular la llegada de los eventos**
>
> El dataset de origen no se indica cuándo llegó cada archivo. Si usaramos el número del nombre como orden de llegada, el watermark tendría que tolerar 60 días para no perder datos. No es un atraso real: es un volcado mezclado.
>
> Para que el streaming se comporte como en un entorno real, esta sección reenvía los eventos a Landing **en el orden en que habrían llegado**: cada evento llega poco después de ocurrir, algunos con atrasos de horas y unos pocos de días (para mostrar el manejo de datos tardíos). La regla es fija (semilla 42), así que el resultado es siempre el mismo.
>
> No modifica ningún evento: copia cada línea tal cual. El dataset de la cursada queda intacto. En producción esta sección no existe.

## 1.3 Entrega de los archivos a Landing

Los CSV llegan como foto completa, cada uno en su carpeta `fecha_carga=AAAA-MM-DD/`

In [142]:
import os

FECHA_CARGA = "2025-09-01"    # día siguiente al último evento
t_carga = datetime.strptime(FECHA_CARGA, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp()

for csv in sorted(DATOS_PROVISTOS.glob("*.csv")):
    destino = LANDING / csv.stem / f"fecha_carga={FECHA_CARGA}"
    destino.mkdir(parents=True, exist_ok=True)
    shutil.copy2(csv, destino / csv.name)          # copia sin modificar
    os.utime(destino / csv.name, (t_carga, t_carga))  # fecha de llegada = fecha de carga

## 1.4 Cuándo llega cada evento

Momento de llegada = momento del evento + un atraso:
- 96 %: entre 0 y 20 minutos (lo normal).
- 3,9 %: entre 1 y 6 horas.
- 0,1 %: entre 3 y 10 días (muy atrasados).

In [143]:
import json, random
from datetime import timedelta

# Se leen las líneas originales: se guardan tal cual, solo se mira la fecha
lineas = []
for archivo in sorted((DATOS_PROVISTOS / "usage_events_stream").glob("*.jsonl")):
    for linea in archivo.read_text(encoding="utf-8").splitlines():
        if linea.strip():
            ev = json.loads(linea)
            lineas.append((ev["event_id"], ev["timestamp"], linea))

lineas.sort()                     # orden fijo, para que el sorteo dé siempre lo mismo
azar = random.Random(42)

llegadas = []
for event_id, ts, linea in lineas:
    ocurrio = datetime.strptime(ts, "%Y-%m-%dT%H:%M:%SZ")
    p = azar.random()
    if p < 0.96:
        atraso = timedelta(minutes=azar.uniform(0, 20))
    elif p < 0.999:
        atraso = timedelta(hours=azar.uniform(1, 6))
    else:
        atraso = timedelta(days=azar.uniform(3, 10))
    llegadas.append((ocurrio + atraso, linea))

## 1.5 Archivos de eventos

El sistema de medición manda un archivo cada 10 minutos con lo que llegó en ese lapso. Se eligieron 10 minutos porque es la meta de atraso del monitoreo de FinOps (diseño 1.4): la fuente no puede mandar archivos más espaciados que eso.

La fecha de modificación de cada archivo es el fin de su lapso: Spark toma los archivos en ese orden.

In [144]:
MINUTOS_POR_ARCHIVO = 10
carpeta_eventos = LANDING / "usage_events_stream"
carpeta_eventos.mkdir(parents=True, exist_ok=True)

# Agrupar por lapso de 10 minutos según el momento de llegada
bloques = {}
for llego, linea in sorted(llegadas):
    inicio = llego.replace(minute=llego.minute // MINUTOS_POR_ARCHIVO * MINUTOS_POR_ARCHIVO,
                           second=0, microsecond=0)
    bloques.setdefault(inicio, []).append(linea)

# Un archivo por lapso
for inicio, contenido in sorted(bloques.items()):
    ruta = carpeta_eventos / f"events_{inicio:%Y%m%dT%H%M}.jsonl"
    ruta.write_text("\n".join(contenido) + "\n", encoding="utf-8")
    t = (inicio + timedelta(minutes=MINUTOS_POR_ARCHIVO)).replace(tzinfo=timezone.utc).timestamp()
    os.utime(ruta, (t, t))

print(f"{len(llegadas)} eventos en {len(bloques)} archivos")

43200 eventos en 8601 archivos


# 2. Landing

Reglas (diseño 7.1 y 7.6):
- No se modifica nada; formato original (CSV y JSONL).
- Se registra cada archivo recibido: nombre, fecha de llegada y tamaño.

## 2.1 Registro de llegadas

In [145]:
llegadas_landing = [
    {"tabla": f.relative_to(LANDING).parts[0],
     "archivo": f.name,
     "kb": round(f.stat().st_size / 1024, 1),
     "llegada": datetime.fromtimestamp(f.stat().st_mtime, timezone.utc).strftime("%Y-%m-%d %H:%M")}
    for f in sorted(LANDING.rglob("*")) if f.is_file()
]
registro_llegadas = pd.DataFrame(llegadas_landing)

# Resumen por tabla
registro_llegadas.groupby("tabla").agg(archivos=("archivo", "count"), kb=("kb", "sum"),
                                       primera=("llegada", "min"), ultima=("llegada", "max"))

,archivos,kb,primera,ultima
tabla,,,,
billing_monthly,1,15.5,2025-09-01 00:00,2025-09-01 00:00
customers_orgs,1,7.6,2025-09-01 00:00,2025-09-01 00:00
marketing_touches,1,99.6,2025-09-01 00:00,2025-09-01 00:00
nps_surveys,1,3.9,2025-09-01 00:00,2025-09-01 00:00
resources,1,35.8,2025-09-01 00:00,2025-09-01 00:00
support_tickets,1,69.3,2025-09-01 00:00,2025-09-01 00:00
usage_events_stream,8601,12676.5,2025-07-03 00:20,2025-09-03 21:40
users,1,69.0,2025-09-01 00:00,2025-09-01 00:00


# 3. Bronze

Reglas de Bronze (diseño 7.1 y 7.7):
- Mismo grano que Landing. **Todavía no se limpia:** las filas con problemas siguen; Silver decide qué hacer con ellas.
- Tipos fijos con esquema explícito, según el diccionario ("tipo origen → objetivo").
- Columnas `ingest_ts` (cuándo se cargó) y `source_file` (de qué archivo vino).
- Sin duplicados por la clave.
- Parquet, con las particiones de D05.
- Pasa a Silver si se leyó completo y se contaron las filas.

## 3.1 Maestros: esquemas

Se cargan `customers_orgs`, `users` y `billing_monthly` (los tres que pide la consigna) y `resources`, que Silver necesita para controlar a qué cliente pertenece cada recurso.

In [146]:
ESQUEMAS = {
    "customers_orgs": StructType([
        StructField("org_id", StringType()),
        StructField("org_name", StringType()),
        StructField("industry", StringType()),
        StructField("hq_region", StringType()),
        StructField("plan_tier", StringType()),
        StructField("is_enterprise", BooleanType()),
        StructField("signup_date", DateType()),
        StructField("sales_rep", StringType()),
        StructField("lifecycle_stage", StringType()),
        StructField("marketing_source", StringType()),
        StructField("nps_score", DoubleType()),     # llega "-3.0": se lee decimal y se pasa a entero
    ]),
    "users": StructType([
        StructField("user_id", StringType()),
        StructField("org_id", StringType()),
        StructField("email", StringType()),
        StructField("role", StringType()),
        StructField("active", BooleanType()),
        StructField("created_at", DateType()),
        StructField("last_login", DateType()),
    ]),
    "resources": StructType([
        StructField("resource_id", StringType()),
        StructField("org_id", StringType()),
        StructField("service", StringType()),
        StructField("region", StringType()),
        StructField("created_at", DateType()),
        StructField("state", StringType()),
        StructField("tags_json", StringType()),     # se guarda crudo; se desarma en Silver
    ]),
    "billing_monthly": StructType([
        StructField("invoice_id", StringType()),
        StructField("org_id", StringType()),
        StructField("month", DateType()),
        StructField("subtotal", DoubleType()),
        StructField("credits", DoubleType()),
        StructField("taxes", DoubleType()),
        StructField("currency", StringType()),
        StructField("exchange_rate_to_usd", DoubleType()),
    ]),
}

# Clave para deduplicar, partición (D05) y conversiones del diccionario
MAESTROS = {
    "customers_orgs":  {"clave": "org_id",      "particion": None,            "a_entero": ["nps_score"]},
    "users":           {"clave": "user_id",     "particion": None,            "a_entero": []},
    "resources":       {"clave": "resource_id", "particion": None,            "a_entero": []},
    "billing_monthly": {"clave": "invoice_id",  "particion": "billing_month", "a_entero": []},
}

## 3.2 Maestros: Landing → Bronze

- `PERMISSIVE` + `_corrupt_record`: si una fila no respeta el esquema, no corta el proceso; el texto original queda en esa columna y la fila sigue (Bronze no limpia).
- `escape`: `tags_json` trae comillas dobles adentro (`"[""env:prod""]"`). Sin esta opción Spark cortaba la fila en la coma interna; el control de `_corrupt_record` fue lo que lo detectó (211 recursos mal leídos).
- `overwrite`: si se vuelve a correr, reemplaza en lugar de agregar filas.
- La carpeta `fecha_carga=…` de Landing se convierte sola en una columna: queda registrado de qué carga vino cada fila.

In [147]:
def leer_maestro(tabla):
    esquema = StructType(ESQUEMAS[tabla].fields + [StructField("_corrupt_record", StringType())])
    return (spark.read
            .option("header", True)
            .option("escape", '"')
            .option("mode", "PERMISSIVE")
            .option("columnNameOfCorruptRecord", "_corrupt_record")
            .schema(esquema)
            .csv(str(LANDING / tabla))                      # todas las fecha_carga de la tabla
            .withColumn("ingest_ts", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_name")))


def bronze_maestro(tabla):
    conf = MAESTROS[tabla]
    df = leer_maestro(tabla).cache()
    leidas = df.count()

    # Tipos del diccionario
    for col in conf["a_entero"]:
        df = df.withColumn(col, F.col(col).cast("int"))

    # Sin duplicados por la clave
    bronze = df.dropDuplicates([conf["clave"]])

    # Escritura en Parquet; billing particionado por mes facturado
    if conf["particion"] == "billing_month":
        (bronze.withColumn("billing_month", F.date_format("month", "yyyy-MM"))
               .write.mode("overwrite").partitionBy("billing_month").parquet(str(BRONZE / tabla)))
    else:
        bronze.write.mode("overwrite").parquet(str(BRONZE / tabla))

    # Control: se leyó todo y no quedaron claves repetidas
    escrito = spark.read.parquet(str(BRONZE / tabla))
    escritas = escrito.count()
    claves = escrito.select(conf["clave"]).distinct().count()
    assert escritas == claves, f"{tabla}: claves repetidas en Bronze"

    anotar("3 bronze", tabla, leidas=leidas,
           corruptas=df.filter(F.col("_corrupt_record").isNotNull()).count(),
           duplicadas=leidas - escritas, escritas=escritas,
           kb=tamano_kb(BRONZE / tabla), ruta=str(BRONZE / tabla))
    df.unpersist()

In [148]:
for tabla in MAESTROS:
    bronze_maestro(tabla)

ver_registro("3 bronze")

,momento,etapa,tabla,leidas,corruptas,duplicadas,escritas,kb,ruta
0,2026-11-25 17:32:26,3 bronze,customers_orgs,80,0,0,80,7.2,/content/lake/bronze/customers_orgs
1,2026-11-25 17:32:27,3 bronze,users,800,0,0,800,23.7,/content/lake/bronze/users
2,2026-11-25 17:32:28,3 bronze,resources,400,0,0,400,12.1,/content/lake/bronze/resources
3,2026-11-25 17:32:29,3 bronze,billing_monthly,240,0,0,240,21.5,/content/lake/bronze/billing_monthly


## 3.3 Maestros: verificación de tipos y particiones

In [149]:
spark.read.parquet(str(BRONZE / "customers_orgs")) \
     .select("org_id", "is_enterprise", "signup_date", "nps_score", "fecha_carga", "ingest_ts", "source_file") \
     .show(3, truncate=False)

spark.read.parquet(str(BRONZE / "billing_monthly")).groupBy("billing_month").count().orderBy("billing_month").show()

+------------+-------------+-----------+---------+-----------+--------------------------+------------------+
|org_id      |is_enterprise|signup_date|nps_score|fecha_carga|ingest_ts                 |source_file       |
+------------+-------------+-----------+---------+-----------+--------------------------+------------------+
|org_0lvsnujz|false        |2025-05-19 |-1       |2025-09-01 |2026-11-25 12:34:46.975962|customers_orgs.csv|
|org_0lzjjege|false        |2025-07-02 |-7       |2025-09-01 |2026-11-25 12:34:46.975962|customers_orgs.csv|
|org_1n13jcat|true         |2025-07-02 |34       |2025-09-01 |2026-11-25 12:34:46.975962|customers_orgs.csv|
+------------+-------------+-----------+---------+-----------+--------------------------+------------------+
only showing top 3 rows
+-------------+-----+
|billing_month|count|
+-------------+-----+
|      2025-06|   80|
|      2025-07|   80|
|      2025-08|   80|
+-------------+-----+



## 3.4 Eventos: esquema

`value` llega a veces como número y a veces como texto (`"107.0"`): se lee como texto y se convierte en el paso siguiente, guardando el original.

In [150]:
ESQUEMA_EVENTOS = StructType([
    StructField("event_id", StringType()),
    StructField("timestamp", TimestampType()),
    StructField("org_id", StringType()),
    StructField("resource_id", StringType()),
    StructField("service", StringType()),
    StructField("region", StringType()),
    StructField("metric", StringType()),
    StructField("value", StringType()),
    StructField("unit", StringType()),
    StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()),        # solo v2
    StructField("genai_tokens", LongType()),       # solo v2 y servicio genai
    StructField("_corrupt_record", StringType()),  # filas que no respetan el esquema
])

## 3.5 Eventos: tipos y metadatos de Bronze

La misma función se usa en el stream y en el control de completitud (3.8).

In [151]:
def tipar_eventos(df):
    return (df
        .withColumnRenamed("timestamp", "event_ts")
        .withColumn("value_raw", F.col("value"))                # valor original, para trazabilidad
        .withColumn("value", F.col("value").cast("double"))     # tipo fijo
        .withColumn("event_date", F.to_date("event_ts"))        # día del evento en UTC (D11)
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name")))

> **Decisión D16 · Watermark de 1 día**
>
> El watermark es "el event time más nuevo visto − tolerancia": compara la fecha de los eventos entre sí, no con la hora de procesamiento. Un evento más viejo que ese límite se considera tardío y el stream lo descarta.
>
> Con 1 día entran los atrasos de horas y quedan afuera solo los muy atrasados. También define cuánto tiempo recuerda Spark los `event_id` para deduplicar. Los descartados no se pierden: en 3.8 se detectan y van a cuarentena con su motivo (D07).

## 3.6 Eventos: stream Landing → Bronze

- `maxFilesPerTrigger`: 720 archivos por micro-batch, o sea 5 días de archivos. Estamos poniendo al día 60 días de historia de una vez; en producción, con un trigger cada pocos minutos, cada micro-batch traería solo los archivos nuevos (uno o dos).
- `trigger(availableNow=True)`: procesa lo pendiente y termina; es la forma de correr streaming como proceso programado (clase 13). En producción sería `processingTime` cada pocos minutos.
- Checkpoint: Spark anota qué archivos ya procesó; si se vuelve a correr, no los lee de nuevo.
- No se usa `cleanSource`, que movería los archivos de Landing.

In [152]:
WATERMARK = "1 day"
ARCHIVOS_POR_TANDA = 20

# Fuente: la carpeta de Landing, leída como stream
eventos_stream = (spark.readStream
    .schema(ESQUEMA_EVENTOS)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("maxFilesPerTrigger", ARCHIVOS_POR_TANDA)
    .json(str(LANDING / "usage_events_stream")))

# Transformación: tipos, watermark y deduplicación por event_id
bronze_eventos = (tipar_eventos(eventos_stream)
    .withWatermark("event_ts", WATERMARK)
    .dropDuplicatesWithinWatermark(["event_id"])
    .repartition("event_date"))                 # un archivo por día y por micro-batch

# Destino: Parquet particionado por día, con checkpoint
query_bronze = (bronze_eventos.writeStream
    .format("parquet")
    .option("path", str(BRONZE / "usage_events"))
    .option("checkpointLocation", str(CHECKPOINTS / "bronze_usage_events"))
    .partitionBy("event_date")
    .outputMode("append")
    .trigger(availableNow=True)
    .queryName("bronze_usage_events")
    .start())

query_bronze.awaitTermination()

## 3.7 Eventos: progreso de los micro-batches

El watermark avanza con las fechas de los eventos que van llegando.

In [153]:
progreso = [
    {"batch": p["batchId"], "filas": p["numInputRows"], "watermark": p["eventTime"].get("watermark")}
    for p in query_bronze.recentProgress if p["numInputRows"] > 0
]

if progreso:
    display(pd.DataFrame(progreso))
else:
    print("Sin archivos nuevos: el checkpoint ya los tenía procesados.")

,batch,filas,watermark
0,332,84,2025-08-17T08:04:00.000Z
1,333,112,2025-08-17T11:19:00.000Z
2,334,96,2025-08-17T14:45:00.000Z
3,335,110,2025-08-17T18:04:00.000Z
4,336,109,2025-08-17T21:14:00.000Z
...,...,...,...
94,426,109,2025-08-30T11:41:00.000Z
95,427,100,2025-08-30T14:56:00.000Z
96,428,116,2025-08-30T18:17:00.000Z
97,429,69,2025-08-30T21:55:00.000Z


## 3.8 Control de completitud

Criterio de éxito "sin pérdidas": todo evento que llegó a Landing está en Bronze o en cuarentena. Los que faltan en Bronze son los que el watermark descartó: van a cuarentena con su motivo.

In [154]:
landing_ev = tipar_eventos(spark.read.schema(ESQUEMA_EVENTOS).json(str(LANDING / "usage_events_stream")))
bronze_ev  = spark.read.parquet(str(BRONZE / "usage_events"))

# Eventos de Landing que no están en Bronze -> cuarentena (regla WM-01)
(landing_ev.join(bronze_ev.select("event_id"), "event_id", "left_anti")
    .withColumn("regla", F.lit("WM-01"))
    .withColumn("motivo", F.lit(f"llegó después del watermark ({WATERMARK})"))
    .withColumn("quarantine_ts", F.current_timestamp())
    .write.mode("overwrite").parquet(str(QUARANTINE / "usage_events_late")))
tardios = spark.read.parquet(str(QUARANTINE / "usage_events_late"))

n_landing, n_bronze, n_tardios = landing_ev.count(), bronze_ev.count(), tardios.count()
n_ids = bronze_ev.select("event_id").distinct().count()

assert n_landing == n_bronze + n_tardios, "Se perdieron eventos entre Landing y Bronze"
assert n_ids == n_bronze, "Hay event_id repetidos en Bronze"

anotar("3 bronze", "usage_events", leidas=n_landing,
       corruptas=bronze_ev.filter(F.col("_corrupt_record").isNotNull()).count(),
       duplicadas=n_bronze - n_ids, tardios_cuarentena=n_tardios, escritas=n_bronze,
       kb=tamano_kb(BRONZE / "usage_events"), ruta=str(BRONZE / "usage_events"))

tardios.select("event_id", "event_ts", "source_file", "regla", "motivo").orderBy("event_ts").show(5, truncate=False)

+----------------+-------------------+--------------------------+-----+-----------------------------------+
|event_id        |event_ts           |source_file               |regla|motivo                             |
+----------------+-------------------+--------------------------+-----+-----------------------------------+
|evt_8lwi8aw6uf7l|2025-07-04 14:10:00|events_20250709T0500.jsonl|WM-01|llegó después del watermark (1 day)|
|evt_0mga3z0ezus1|2025-07-06 08:35:00|events_20250714T0920.jsonl|WM-01|llegó después del watermark (1 day)|
|evt_56z4as1lj11z|2025-07-06 08:47:00|events_20250715T1250.jsonl|WM-01|llegó después del watermark (1 day)|
|evt_wijx2k24ky19|2025-07-06 13:45:00|events_20250716T0530.jsonl|WM-01|llegó después del watermark (1 day)|
|evt_mf7ifzzvpw9t|2025-07-08 23:39:00|events_20250718T1200.jsonl|WM-01|llegó después del watermark (1 day)|
+----------------+-------------------+--------------------------+-----+-----------------------------------+
only showing top 5 rows


## 3.9 Resumen de Bronze

In [155]:
ver_registro("3 bronze")

,momento,etapa,tabla,leidas,corruptas,duplicadas,escritas,kb,ruta,tardios_cuarentena
0,2026-11-25 17:32:26,3 bronze,customers_orgs,80,0,0,80,7.2,/content/lake/bronze/customers_orgs,
1,2026-11-25 17:32:27,3 bronze,users,800,0,0,800,23.7,/content/lake/bronze/users,
2,2026-11-25 17:32:28,3 bronze,resources,400,0,0,400,12.1,/content/lake/bronze/resources,
3,2026-11-25 17:32:29,3 bronze,billing_monthly,240,0,0,240,21.5,/content/lake/bronze/billing_monthly,
4,2026-11-25 17:36:04,3 bronze,usage_events,43200,0,0,43171,5775.0,/content/lake/bronze/usage_events,29.0


# 4. Silver

Reglas de Silver (diseño 7.1 y diccionario, sección 5):
- **Nada se borra:** cada registro se marca (columna `flag_…`) o va a cuarentena (`silver/quarantine/`) con la regla y el motivo.
- Eventos v1 y v2 en una sola tabla.
- Unión de los eventos con los maestros limpios por `org_id` y `resource_id`.
- Sin SCD (D09): los maestros son la foto actual.
- Pasa a Gold si se aplicaron todas las reglas y se registró cuántas filas se marcaron y cuántas fueron a cuarentena.

Los maestros se procesan en batch y los eventos en streaming, igual que en el diagrama.

## 4.1 Maestros limpios: clientes

| Regla | Qué controla | Si no se cumple |
|---|---|---|
| FMT-01 | La fila respeta el esquema (`_corrupt_record` vacío) | Cuarentena |
| DOM-01 | `nps_score` entre −100 y 100 | Marcar (`flag_nps_out_of_range`) |
| DOM-02 | `plan_tier` dentro de sus valores (free, standard, pro, enterprise) | Cuarentena |

In [156]:
PLANES = ["free", "standard", "pro", "enterprise"]

clientes = (spark.read.parquet(str(BRONZE / "customers_orgs"))
    .withColumn("regla", F.when(F.col("_corrupt_record").isNotNull(), "FMT-01")
                          .when(~F.col("plan_tier").isin(PLANES), "DOM-02"))
    .withColumn("flag_nps_out_of_range", ~F.col("nps_score").between(-100, 100)))

clientes_ok  = clientes.filter(F.col("regla").isNull()).drop("regla", "_corrupt_record")
clientes_cua = clientes.filter(F.col("regla").isNotNull())

clientes_ok.write.mode("overwrite").parquet(str(SILVER / "customers_orgs"))
clientes_cua.withColumn("quarantine_ts", F.current_timestamp()) \
            .write.mode("overwrite").parquet(str(QUARANTINE / "customers_orgs"))

anotar("4 silver", "customers_orgs", leidas=clientes.count(),
       escritas=spark.read.parquet(str(SILVER / "customers_orgs")).count(),
       cuarentena=clientes_cua.count(),
       marcadas=clientes_ok.filter("flag_nps_out_of_range").count(),
       kb=tamano_kb(SILVER / "customers_orgs"), ruta=str(SILVER / "customers_orgs"))

clientes_ok.filter("flag_nps_out_of_range").select("org_id", "org_name", "nps_score", "flag_nps_out_of_range").show()

+------------+--------------+---------+---------------------+
|      org_id|      org_name|nps_score|flag_nps_out_of_range|
+------------+--------------+---------+---------------------+
|org_pac56t4u|Gamma Cloud 37|      101|                 true|
+------------+--------------+---------+---------------------+



Ningún cliente fue a cuarentena y uno quedó marcado: Gamma Cloud 37, con NPS 101. Es el mismo que encontraron en la exploración. Sigue en Silver, porque un NPS raro no invalida al cliente; queda señalado para que nadie lo use en un promedio sin saberlo.

## 4.2 Maestros limpios: recursos

`tags_json` llega como texto con una lista adentro. Acá se desarma en una lista y se sacan dos datos útiles: el ambiente (`env`) y si el recurso guarda datos personales (`pii:true`, para gobierno).

| Regla | Qué controla | Si no se cumple |
|---|---|---|
| FMT-01 | La fila respeta el esquema | Cuarentena |
| RI-01 | El `org_id` existe en clientes | Cuarentena |

In [157]:
recursos = (spark.read.parquet(str(BRONZE / "resources"))
    .join(clientes_ok.select("org_id", F.lit(True).alias("org_existe")), "org_id", "left")
    .withColumn("regla", F.when(F.col("_corrupt_record").isNotNull(), "FMT-01")
                          .when(F.col("org_existe").isNull(), "RI-01"))
    # Etiquetas: texto -> lista
    .withColumn("tags", F.coalesce(F.from_json("tags_json", "ARRAY<STRING>"), F.array().cast("ARRAY<STRING>")))
    .withColumn("env", F.expr("substr(try_element_at(filter(tags, t -> t LIKE 'env:%'), 1), 5)"))   # prod / dev
    .withColumn("has_pii", F.array_contains("tags", "pii:true")))

recursos_ok  = recursos.filter(F.col("regla").isNull()).drop("regla", "_corrupt_record", "org_existe")
recursos_cua = recursos.filter(F.col("regla").isNotNull())

recursos_ok.write.mode("overwrite").parquet(str(SILVER / "resources"))
recursos_cua.withColumn("quarantine_ts", F.current_timestamp()) \
            .write.mode("overwrite").parquet(str(QUARANTINE / "resources"))

anotar("4 silver", "resources", leidas=recursos.count(),
       escritas=spark.read.parquet(str(SILVER / "resources")).count(),
       cuarentena=recursos_cua.count(), kb=tamano_kb(SILVER / "resources"), ruta=str(SILVER / "resources"))

recursos_ok.select("resource_id", "tags_json", "tags", "env", "has_pii").show(4, truncate=False)

+------------+-----------------------------------+-----------------------------+----+-------+
|resource_id |tags_json                          |tags                         |env |has_pii|
+------------+-----------------------------------+-----------------------------+----+-------+
|res_00v3sfav|["backup:daily", "env:dev"]        |[backup:daily, env:dev]      |dev |false  |
|res_019ydq3n|["backup:daily"]                   |[backup:daily]               |NULL|false  |
|res_02grxpud|["team:ml"]                        |[team:ml]                    |NULL|false  |
|res_03njbcfa|["team:ml", "team:core", "env:dev"]|[team:ml, team:core, env:dev]|dev |false  |
+------------+-----------------------------------+-----------------------------+----+-------+
only showing top 4 rows


En la tabla se ve el antes (tags_json, texto) y el después (tags, lista). Si un recurso no tiene la etiqueta env:, el ambiente queda en NULL: no se inventa un valor. Ninguno fue a cuarentena, porque todos los dueños existen (lo vimos en la exploración: "no hay registros huérfanos")

## 4.3 Umbral de anomalías de costo

Los costos normales dependen mucho del servicio y de la métrica: un evento de `requests` de `genai` cuesta unos 14 USD y uno de `cpu_hours` de `networking`, 0,02. Un umbral único no sirve, así que se calcula uno por combinación servicio + métrica a partir del histórico de Bronze.

In [158]:
# Percentiles 50 y 99 del costo por servicio y métrica
umbrales = (spark.read.parquet(str(BRONZE / "usage_events"))
    .groupBy("service", "metric")
    .agg(F.expr("percentile(cost_usd_increment, 0.5)").alias("p50"),
         F.expr("percentile(cost_usd_increment, 0.99)").alias("p99")))

# ¿Cuántos eventos superan k veces el p99 de su grupo?
con_umbral = spark.read.parquet(str(BRONZE / "usage_events")).join(umbrales, ["service", "metric"])
spark.createDataFrame(
    [(k, con_umbral.filter(F.col("cost_usd_increment") > k * F.col("p99")).count()) for k in [1.0, 1.5, 2.0, 3.0, 5.0]],
    ["veces_p99", "eventos_por_encima"]).show()

+---------+------------------+
|veces_p99|eventos_por_encima|
+---------+------------------+
|      1.0|               439|
|      1.5|               151|
|      2.0|               150|
|      3.0|               138|
|      5.0|               126|
+---------+------------------+



> **Decisión D17 · Umbrales de anomalía de costo**
>
> Un evento es anomalía de costo (regla EV-05) si:
> - el costo es menor a −0,01 USD (la consigna lo pide), o
> - el costo supera **2 veces el percentil 99** de su servicio y métrica.
>
> Por encima de 1,5 veces el p99 casi no cambia la cantidad de eventos (entre 150 y 138): los picos forman un grupo aparte, de 10 a 20 veces el costo normal. Con 2 veces quedan adentro todos los picos sin marcar costos apenas altos.
>
> Los umbrales se calculan con el histórico de Bronze y se guardan como tabla de referencia. En producción se recalcularían cada tanto (por ejemplo, una vez por mes).

In [159]:
FACTOR_P99 = 2
umbrales = umbrales.withColumn("cost_spike_threshold", F.col("p99") * FACTOR_P99)
umbrales.write.mode("overwrite").parquet(str(SILVER / "ref_cost_thresholds"))
umbrales.orderBy("service", "metric").show(18)

+----------+----------------+--------------------+--------------------+--------------------+
|   service|          metric|                 p50|                 p99|cost_spike_threshold|
+----------+----------------+--------------------+--------------------+--------------------+
| analytics|       cpu_hours|              0.1282| 0.43633999999999995|  0.8726799999999999|
| analytics|        requests|   8.349450000000001|            12.22035|             24.4407|
| analytics|storage_gb_hours|              0.7016|  2.0397199999999995|   4.079439999999999|
|   compute|       cpu_hours|              0.1398| 0.45919999999999983|  0.9183999999999997|
|   compute|        requests|              9.5204|  14.357271999999996|  28.714543999999993|
|   compute|storage_gb_hours|  0.7678499999999999|  2.6154260000000003|  5.2308520000000005|
|  database|       cpu_hours|              0.0927|  0.2939999999999996|  0.5879999999999992|
|  database|        requests|              5.9529|            8.772838

## 4.4 Reglas de calidad de los eventos

Las reglas del diccionario (sección 5) que aplican a los eventos. Las tres que pide la consigna como mínimo son EV-01, EV-05 y EV-03.

| Regla | Qué controla | Si no se cumple |
|---|---|---|
| FMT-01 | La fila respeta el esquema | Cuarentena |
| EV-01 | `event_id` no vacío (la unicidad se garantiza en Bronze) | Cuarentena |
| EV-02 | `value` convertible a número | Cuarentena |
| EV-04 | El evento tiene `value` o `unit` | Cuarentena |
| RI-01 | El `org_id` existe en clientes | Cuarentena |
| RI-02 | El `resource_id` existe y es del mismo cliente | Cuarentena |
| EV-03 | Si hay `value`, hay `unit` | Completar según la métrica y marcar (`flag_unit_filled`) |
| EV-05 | Costo ≥ −0,01 y sin picos (D17) | Marcar (`flag_cost_negative`, `flag_cost_spike`, `is_cost_anomaly`) |
| EV-06 | `genai_tokens` solo en `genai` y v2 | Marcar (`flag_genai_tokens`) |
| TI-01 | El evento no es anterior a la creación del recurso | Marcar (`flag_before_resource`), no descartar (D08) |

Si un evento no cumple varias reglas de cuarentena, queda registrada la primera de la tabla.

In [160]:
# Unidad que corresponde a cada métrica (EV-03)
UNIDAD = F.create_map(F.lit("requests"), F.lit("count"),
                      F.lit("cpu_hours"), F.lit("hours"),
                      F.lit("storage_gb_hours"), F.lit("gb_hours"))

# Maestros limpios y umbrales: referencia para el join (se leen de Silver)
ref_clientes = spark.read.parquet(str(SILVER / "customers_orgs")).select(
    "org_id", "org_name", "industry", "plan_tier", "hq_region")
ref_recursos = spark.read.parquet(str(SILVER / "resources")).select(
    "resource_id", F.col("org_id").alias("resource_org_id"),
    F.col("created_at").alias("resource_created_at"), F.col("state").alias("resource_state"), "env", "has_pii")
ref_umbrales = spark.read.parquet(str(SILVER / "ref_cost_thresholds")).select("service", "metric", "cost_spike_threshold")


def evaluar_eventos(df):
    """Une con los maestros, aplica las reglas de calidad y calcula las features."""
    return (df
        # Unión con los maestros (enriquecimiento) y con los umbrales
        .join(F.broadcast(ref_clientes), "org_id", "left")
        .join(F.broadcast(ref_recursos), "resource_id", "left")
        .join(F.broadcast(ref_umbrales), ["service", "metric"], "left")

        # Reglas que mandan a cuarentena: la primera que falla
        .withColumn("regla", F.when(F.col("_corrupt_record").isNotNull(), "FMT-01")
            .when(F.col("event_id").isNull(), "EV-01")
            .when(F.col("value_raw").isNotNull() & F.col("value").isNull(), "EV-02")
            .when(F.col("value").isNull() & F.col("unit").isNull(), "EV-04")
            .when(F.col("org_name").isNull(), "RI-01")
            .when(F.col("resource_org_id").isNull() | (F.col("resource_org_id") != F.col("org_id")), "RI-02"))

        # Reglas que marcan
        .withColumn("unit_raw", F.col("unit"))                  # unidad original, para trazabilidad
        .withColumn("flag_unit_filled", F.col("value").isNotNull() & F.col("unit").isNull())
        .withColumn("unit", F.coalesce("unit", UNIDAD[F.col("metric")]))
        .withColumn("flag_cost_negative", F.col("cost_usd_increment") < -0.01)
        .withColumn("flag_cost_spike", F.col("cost_usd_increment") > F.col("cost_spike_threshold"))
        .withColumn("is_cost_anomaly", F.col("flag_cost_negative") | F.col("flag_cost_spike"))
        .withColumn("flag_genai_tokens", F.col("genai_tokens").isNotNull()
                    & ((F.col("service") != "genai") | (F.col("schema_version") != 2)))
        .withColumn("flag_before_resource", F.col("event_date") < F.col("resource_created_at"))

        # Features: una columna por métrica, con el valor solo si el evento es de esa métrica
        .withColumn("cost_usd", F.col("cost_usd_increment"))
        .withColumn("requests", F.when(F.col("metric") == "requests", F.col("value")).otherwise(0.0))
        .withColumn("cpu_hours", F.when(F.col("metric") == "cpu_hours", F.col("value")).otherwise(0.0))
        .withColumn("storage_gb_hours", F.when(F.col("metric") == "storage_gb_hours", F.col("value")).otherwise(0.0))
        .withColumn("genai_tokens", F.coalesce("genai_tokens", F.lit(0)))
        .withColumn("silver_ts", F.current_timestamp()))

Reglas de los eventos

Primero arma tres "tablas de consulta": clientes, recursos y umbrales. Después, la función evaluar_eventos hace tres cosas con cada evento:

* Lo enriquece: busca su cliente (nombre, industria, plan) y su recurso (dueño, fecha de creación, ambiente).
* Decide si va a cuarentena: la columna regla revisa las condiciones en orden y anota la primera que falla. Por ejemplo, value y unit vacíos a la vez dan EV-04. Si ninguna falla, queda vacía.
* Pone las marcas: columnas flag_… que valen verdadero o falso. El evento sigue igual en Silver, pero señalado.

broadcast le dice a Spark que las tablas de consulta son chicas y que mande una copia entera a cada núcleo. Así cada evento se cruza donde está y no hace falta mover los eventos entre nodos (se evita un shuffle).

## 4.5 Features

La consigna pide al menos tres. Se calculan por evento en Silver y se suman por día en Gold, convierte cada evento en columnas fáciles de sumar:

| Feature | De dónde sale |
|---|---|
| `cost_usd` | `cost_usd_increment` (en Gold: `daily_cost_usd`) |
| `requests`, `cpu_hours`, `storage_gb_hours` | `value` según la métrica del evento; 0 si el evento es de otra métrica |
| `genai_tokens` | Solo eventos `genai` v2; 0 en el resto |
| `carbon_kg` | Solo v2. En v1 queda vacío: no se inventa un 0 donde el dato no existía |
| `is_cost_anomaly` | Regla EV-05 con los umbrales de D17 |

## 4.6 Stream Bronze → Silver

Lee `bronze/usage_events` como stream y escribe dos salidas, cada una con su checkpoint:
- **`silver/usage_events`**: eventos válidos, enriquecidos y con sus marcas, particionados por día.
- **`silver/quarantine/usage_events`**: los que no pasaron una regla de cuarentena, con la regla y el motivo.

Los maestros se unen como tablas estáticas (join stream-estático): cada micro-batch usa la foto vigente de Silver.

In [161]:
MOTIVOS = {"FMT-01": "no respeta el esquema", "EV-01": "event_id vacío", "EV-02": "value no convertible a número",
           "EV-04": "sin value ni unit", "RI-01": "org_id inexistente", "RI-02": "resource_id inexistente o de otro cliente"}
motivo = F.create_map(*[F.lit(x) for kv in MOTIVOS.items() for x in kv])

# Fuente: Bronze como stream (mismo esquema que lo escrito en Bronze)
bronze_stream = (spark.readStream
    .schema(spark.read.parquet(str(BRONZE / "usage_events")).schema)
    .option("maxFilesPerTrigger", 30)
    .parquet(str(BRONZE / "usage_events")))

evaluados = evaluar_eventos(bronze_stream)

COLUMNAS_SILVER = [
    "event_id", "event_ts", "event_date", "org_id", "org_name", "industry", "plan_tier", "hq_region",
    "resource_id", "resource_state", "env", "has_pii", "service", "region", "metric", "value", "value_raw", "unit",
    "cost_usd", "requests", "cpu_hours", "storage_gb_hours", "genai_tokens", "carbon_kg", "schema_version",
    "is_cost_anomaly", "flag_cost_negative", "flag_cost_spike", "flag_unit_filled",
    "flag_genai_tokens", "flag_before_resource", "source_file", "ingest_ts", "silver_ts"]

# Salida 1: eventos válidos
q_silver = (evaluados.filter(F.col("regla").isNull()).select(COLUMNAS_SILVER)
    .writeStream.format("parquet")
    .option("path", str(SILVER / "usage_events"))
    .option("checkpointLocation", str(CHECKPOINTS / "silver_usage_events"))
    .partitionBy("event_date")
    .trigger(availableNow=True)
    .queryName("silver_usage_events")
    .start())

# Salida 2: cuarentena
q_cuarentena = (evaluados.filter(F.col("regla").isNotNull())
    .withColumn("motivo", motivo[F.col("regla")])
    .withColumn("quarantine_ts", F.current_timestamp())
    .writeStream.format("parquet")
    .option("path", str(QUARANTINE / "usage_events"))
    .option("checkpointLocation", str(CHECKPOINTS / "silver_quarantine_usage_events"))
    .trigger(availableNow=True)
    .queryName("silver_quarantine_usage_events")
    .start())

q_silver.awaitTermination()
q_cuarentena.awaitTermination()

## 4.7 Control y muestras de cuarentena

Criterios de éxito "sin pérdidas" y "sin duplicados": Bronze = Silver + cuarentena, y cada `event_id` aparece una sola vez.

In [162]:
bronze_ev = spark.read.parquet(str(BRONZE / "usage_events"))
silver_ev = spark.read.parquet(str(SILVER / "usage_events"))
cuarentena_ev = spark.read.parquet(str(QUARANTINE / "usage_events"))

n_bronze, n_silver, n_cua = bronze_ev.count(), silver_ev.count(), cuarentena_ev.count()

assert n_bronze == n_silver + n_cua, "Se perdieron eventos entre Bronze y Silver"
assert silver_ev.select("event_id").distinct().count() == n_silver, "Hay event_id repetidos en Silver"

marcas = {c: silver_ev.filter(F.col(c)).count() for c in
          ["flag_unit_filled", "flag_cost_negative", "flag_cost_spike", "is_cost_anomaly",
           "flag_genai_tokens", "flag_before_resource"]}

con_marca = silver_ev.filter(" OR ".join(marcas)).count()     # eventos con al menos una marca

anotar("4 silver", "usage_events", leidas=n_bronze, escritas=n_silver, cuarentena=n_cua,
       marcadas=con_marca, kb=tamano_kb(SILVER / "usage_events"),
       ruta=str(SILVER / "usage_events"))

print(f"Bronze {n_bronze} = Silver {n_silver} + cuarentena {n_cua}")
pd.DataFrame([marcas], index=["eventos marcados"]).T

Bronze 43171 = Silver 43134 + cuarentena 37


,eventos marcados
flag_unit_filled,2038
flag_cost_negative,210
flag_cost_spike,150
is_cost_anomaly,360
flag_genai_tokens,0
flag_before_resource,7354


In [163]:
print("Cuarentena por regla:")
cuarentena_ev.groupBy("regla", "motivo").count().orderBy("regla").show(truncate=False)

print("Muestra:")
cuarentena_ev.select("event_id", "event_ts", "metric", "value_raw", "unit_raw", "cost_usd_increment", "regla", "motivo") \
             .show(5, truncate=False)

Cuarentena por regla:
+-----+-----------------+-----+
|regla|motivo           |count|
+-----+-----------------+-----+
|EV-04|sin value ni unit|37   |
+-----+-----------------+-----+

Muestra:
+----------------+-------------------+----------------+---------+--------+------------------+-----+-----------------+
|event_id        |event_ts           |metric          |value_raw|unit_raw|cost_usd_increment|regla|motivo           |
+----------------+-------------------+----------------+---------+--------+------------------+-----+-----------------+
|evt_736bk4jqf0c9|2025-07-23 12:57:00|requests        |NULL     |NULL    |10.8702           |EV-04|sin value ni unit|
|evt_cqhb7ms7cs4b|2025-07-25 13:17:00|cpu_hours       |NULL     |NULL    |0.0342            |EV-04|sin value ni unit|
|evt_htr2s0cewzbd|2025-07-23 21:45:00|requests        |NULL     |NULL    |8.4862            |EV-04|sin value ni unit|
|evt_wukiz0ia77y9|2025-08-06 11:33:00|requests        |NULL     |NULL    |6.7295            |EV-04|s

Cuarentena: 37 eventos, todos por EV-04. En la muestra se ve value_raw y unit_raw en NULL: sin cantidad y sin unidad no hay forma de saber cuánto se consumió. Es el mismo número (37) que vimos en la exploración

In [164]:
print("Anomalías de costo (muestra):")
silver_ev.filter("is_cost_anomaly") \
    .select("event_id", "org_name", "service", "metric", "cost_usd", "flag_cost_negative", "flag_cost_spike") \
    .orderBy(F.desc("cost_usd")).show(5, truncate=False)

print("Unidad completada (EV-03):")
silver_ev.filter("flag_unit_filled").select("event_id", "metric", "value", "unit", "flag_unit_filled").show(3)

print("Eventos por versión de esquema (v1 y v2 en una sola tabla):")
silver_ev.groupBy("schema_version").agg(F.count("*").alias("eventos"),
                                        F.count("carbon_kg").alias("con_carbon_kg")).orderBy("schema_version").show()

Anomalías de costo (muestra):
+----------------+----------------+-------+--------+--------+------------------+---------------+
|event_id        |org_name        |service|metric  |cost_usd|flag_cost_negative|flag_cost_spike|
+----------------+----------------+-------+--------+--------+------------------+---------------+
|evt_j1oqzbmsh98m|Apex Cloud 17   |genai  |requests|317.4308|false             |true           |
|evt_rv7enc8vnthi|Vertex AI 53    |genai  |requests|307.0602|false             |true           |
|evt_qnvh97deajfh|Apex Cloud 34   |compute|requests|302.3459|false             |true           |
|evt_mftsdyy07bsq|Nova Tech 61    |genai  |requests|300.396 |false             |true           |
|evt_yqdtb1nzsqfn|Omega Digital 40|compute|requests|261.3299|false             |true           |
+----------------+----------------+-------+--------+--------+------------------+---------------+
only showing top 5 rows
Unidad completada (EV-03):
+----------------+----------------+------+----

Anomalías: los más caros son de genai y compute con requests, de 260 a 317 USD, cuando lo típico en esos grupos es de 9 a 14. Esos son los casos que FinOps tendría que ver.

Versiones: v1 y v2 quedaron en la misma tabla. con_carbon_kg vale 0 en v1 porque ese dato no existía en esa versión, y por eso no se completa.

## 4.8 Resumen de Silver

In [165]:
ver_registro("4 silver")

,momento,etapa,tabla,leidas,escritas,kb,ruta,cuarentena,marcadas
5,2026-11-25 17:36:05,4 silver,customers_orgs,80,80,7.2,/content/lake/silver/customers_orgs,0.0,1.0
6,2026-11-25 17:36:07,4 silver,resources,400,400,13.5,/content/lake/silver/resources,0.0,
7,2026-11-25 17:36:54,4 silver,usage_events,43171,43134,4473.8,/content/lake/silver/usage_events,37.0,9352.0


| Regla | Cómo se cumple |
|---|---|
| Limpiar eventos y al menos un maestro (consigna) | Eventos (4.6) y dos maestros: clientes (4.1) y recursos (4.2) |
| Joins de enriquecimiento (consigna) | Eventos + clientes + recursos (4.4) |
| Al menos tres features (consigna) | `cost_usd`, `requests`, `cpu_hours`, `storage_gb_hours`, `genai_tokens`, `carbon_kg`, `is_cost_anomaly` (4.5) |
| Al menos tres reglas de calidad y cuarentena con muestras (consigna) | Diez reglas (4.1, 4.2, 4.4); muestras en 4.7 |
| Nada se borra | Lo que no pasa va a `silver/quarantine/` con regla y motivo; `assert` Bronze = Silver + cuarentena |
| Eventos v1 y v2 en una sola tabla | `carbon_kg` vacío en v1 (4.7) |
| Fechas fuera de orden: marcar, no descartar (D08) | `flag_before_resource` |

**Fuera de esta entrega:** Silver de `users` y `billing_monthly` (reglas BI-01 a BI-03) y de tickets, marketing y NPS. Quedan en el backlog para el final (P3 y P4).

# 5. Gold

Tablas resumidas para los consumidores, una por familia de preguntas (diseño 7.1):
- Grano explícito: qué representa cada fila.
- Sale solo de Silver.
- Pasa a Cassandra si los totales coinciden con Silver (diseño 7.7).

En esta entrega se crea `org_daily_usage_by_service` (FinOps), que responde P1 y P2. Es un proceso **batch nocturno** (D02): lee los eventos enriquecidos de Silver y suma por día. Por ser batch no lleva checkpoint (eso es de las consultas de streaming); su protección contra duplicados es reescribir los días que procesa.

> **Decisión D18 · Las anomalías se suman al costo diario**
>
> Los eventos marcados como anomalía (EV-05) **se incluyen** en `daily_cost_usd`: el costo existió y FinOps tiene que ver el gasto completo. Al lado se informa cuántos eventos anómalos hubo y cuánto suman (`anomaly_events`, `anomaly_cost_usd`), así se pueden separar si hace falta.

## 5.1 `org_daily_usage_by_service`

**Grano:** una fila por cliente, día y servicio.

Es el cálculo del diseño (sección 9): el `groupBy` es el shuffle y la suma es el reduce. Spark hace sumas parciales en cada núcleo antes del shuffle, así viaja una fila por clave y no un evento por fila.

In [166]:
silver_ev = spark.read.parquet(str(SILVER / "usage_events"))

gold_calculado = (silver_ev
    .groupBy("org_id", "org_name", F.col("event_date").alias("usage_date"), "service")
    .agg(F.round(F.sum("cost_usd"), 4).alias("daily_cost_usd"),
         F.sum("requests").alias("requests"),
         F.round(F.sum("cpu_hours"), 4).alias("cpu_hours"),
         F.round(F.sum("storage_gb_hours"), 4).alias("storage_gb_hours"),
         F.sum("genai_tokens").alias("genai_tokens"),
         F.round(F.sum("carbon_kg"), 6).alias("carbon_kg"),          # vacío si todos los eventos del día son v1
         F.count("*").alias("events"),
         F.sum(F.col("is_cost_anomaly").cast("int")).alias("anomaly_events"),
         F.round(F.sum(F.when(F.col("is_cost_anomaly"), F.col("cost_usd")).otherwise(0)), 4).alias("anomaly_cost_usd"))
    .withColumn("gold_ts", F.current_timestamp()))

# Reescribe solo los días que se procesan (dynamic): volver a correrlo no duplica filas
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
gold_calculado.write.mode("overwrite").partitionBy("usage_date").parquet(str(GOLD / "org_daily_usage_by_service"))

gold_uso = spark.read.parquet(str(GOLD / "org_daily_usage_by_service"))
gold_uso.orderBy("usage_date", "org_id", "service").show(5)

+------------+--------------+----------+--------------+--------+---------+----------------+------------+---------+------+--------------+----------------+--------------------+----------+
|      org_id|      org_name|   service|daily_cost_usd|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|events|anomaly_events|anomaly_cost_usd|             gold_ts|usage_date|
+------------+--------------+----------+--------------+--------+---------+----------------+------------+---------+------+--------------+----------------+--------------------+----------+
|org_0lvsnujz| Nova Cloud 45| analytics|        8.8739|   116.0|   4.6168|         12.7808|           0|     NULL|     3|             0|             0.0|2026-11-25 17:36:...|2025-07-03|
|org_0lvsnujz| Nova Cloud 45|  database|        0.5723|     0.0|   1.7515|         14.7891|           0|     NULL|     3|             0|             0.0|2026-11-25 17:36:...|2025-07-03|
|org_0lvsnujz| Nova Cloud 45|   storage|        5.5542|   255.0|      

El código toma todos los eventos de Silver y los agrupa por cliente, día y servicio. para cada grupo calcula:
* Suma del costo del día
* Suma cada feature
* anomaly_events y anomaly_cost_usd: cuántos de esos eventos eran anomalías y cuánto sumaron (D18).

Después lo escribe en Parquet partido por día. La línea partitionOverwriteMode = dynamic hace que, si se vuelve a correr, Spark reemplace los días que procesa en vez de agregar filas de nuevo. Es la protección contra duplicados del batch nocturno: si el proceso de anoche falló y se repite, el día queda igual, no doble

Muestra:

* events = 3: cada fila junta pocos eventos. Hay 720 eventos por día repartidos entre 80 clientes y 6 servicios, así que cada combinación tiene muy pocos. En producción serían miles.
* requests = 0.0 en database: ese día los eventos de database de ese cliente fueron de CPU y almacenamiento, no de requests. No falta el dato: es cero de verdad.
* carbon_kg = NULL: es 3 de julio, período v1, donde el carbono no se medía. Se deja vacío en vez de poner 0 (lo mismo que en Silver).
anomaly_events = 0: esas filas no tuvieron anomalías.

## 5.2 Validación

Se controla antes de pasar a Cassandra:
1. **Grano:** no hay dos filas con el mismo cliente, día y servicio.
2. **Totales Gold = Silver** (criterio "números correctos"): costo, requests y cantidad de eventos.
3. **Cálculo de referencia con MapReduce:** el mismo resultado calculado a mano con `map` y `reduceByKey` sobre RDD, como en el diagrama del diseño. Tiene que dar igual, fila por fila.

In [167]:
# 1. Grano
filas = gold_uso.count()
claves = gold_uso.select("org_id", "usage_date", "service").distinct().count()
assert filas == claves, "Hay filas repetidas para el mismo cliente, día y servicio"

# 2. Totales Gold vs Silver
tot_silver = silver_ev.agg(F.sum("cost_usd").alias("costo"), F.sum("requests").alias("requests"),
                           F.count("*").alias("eventos")).first()
tot_gold = gold_uso.agg(F.sum("daily_cost_usd").alias("costo"), F.sum("requests").alias("requests"),
                        F.sum("events").alias("eventos")).first()

comparacion = pd.DataFrame({"silver": tot_silver.asDict(), "gold": tot_gold.asDict()})
comparacion["diferencia"] = (comparacion["gold"] - comparacion["silver"]).round(2)
assert (comparacion["diferencia"].abs() < 0.5).all(), "Los totales de Gold no coinciden con Silver"
comparacion

,silver,gold,diferencia
costo,147202.518,147202.518,-0.0
requests,2292817.000,2292817.000,0.0
eventos,43134.000,43134.000,0.0


In [168]:
# 3. MapReduce a mano: map -> (clave, valor) ; reduceByKey -> suma por clave
pares = silver_ev.select("org_id", "event_date", "service", "cost_usd", "requests").rdd.map(
    lambda e: ((e.org_id, e.event_date, e.service), (e.cost_usd, e.requests or 0.0)))
reducido = pares.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

referencia = reducido.map(lambda kv: (kv[0][0], kv[0][1], kv[0][2], round(kv[1][0], 4), kv[1][1])) \
                     .toDF(["org_id", "usage_date", "service", "costo_ref", "requests_ref"])

distintas = (gold_uso.join(referencia, ["org_id", "usage_date", "service"], "full")
             .filter((F.abs(F.col("daily_cost_usd") - F.col("costo_ref")) > 0.001)
                     | (F.col("requests") != F.col("requests_ref"))
                     | F.col("daily_cost_usd").isNull() | F.col("costo_ref").isNull())
             .count())
assert distintas == 0, "El cálculo de Gold no coincide con el de referencia"
print(f"Filas de Gold: {filas} | Filas distintas contra MapReduce: {distintas}")

anotar("5 gold", "org_daily_usage_by_service", leidas=tot_silver["eventos"], escritas=filas,
       dias=gold_uso.select("usage_date").distinct().count(),
       kb=tamano_kb(GOLD / "org_daily_usage_by_service"), ruta=str(GOLD / "org_daily_usage_by_service"))

Filas de Gold: 11050 | Filas distintas contra MapReduce: 0


## 5.3 Vista rápida: P1 y P2 sobre Gold

Antes de cargar Cassandra, las dos preguntas resueltas con Spark sobre Parquet. Sirven de referencia para comparar con lo que devuelva Cassandra en la sección 6.

In [169]:
ORG_EJEMPLO = gold_uso.groupBy("org_id").agg(F.sum("daily_cost_usd").alias("c")).orderBy(F.desc("c")).first()["org_id"]
print("Cliente de ejemplo:", ORG_EJEMPLO)

# P1: costos y requests diarios por servicio en un rango de fechas
gold_uso.filter((F.col("org_id") == ORG_EJEMPLO) & F.col("usage_date").between("2025-08-01", "2025-08-03")) \
        .select("usage_date", "service", "daily_cost_usd", "requests", "anomaly_events") \
        .orderBy("usage_date", "service").show()

# P2: top 5 servicios por costo acumulado en los últimos 14 días
ultimo_dia = gold_uso.agg(F.max("usage_date")).first()[0]
gold_uso.filter((F.col("org_id") == ORG_EJEMPLO) & (F.col("usage_date") > F.date_sub(F.lit(ultimo_dia), 14))) \
        .groupBy("service").agg(F.round(F.sum("daily_cost_usd"), 2).alias("costo_14_dias")) \
        .orderBy(F.desc("costo_14_dias")).show(5)

Cliente de ejemplo: org_53lc58dr
+----------+----------+--------------+--------+--------------+
|usage_date|   service|daily_cost_usd|requests|anomaly_events|
+----------+----------+--------------+--------+--------------+
|2025-08-01|  database|        1.5839|     0.0|             0|
|2025-08-01|     genai|       15.6442|   131.0|             0|
|2025-08-01|networking|        4.3094|   375.0|             0|
|2025-08-02| analytics|       32.7553|   478.0|             0|
|2025-08-02|  database|        1.5619|     0.0|             0|
|2025-08-02|networking|        2.2607|   237.0|             0|
|2025-08-02|   storage|        4.8398|   255.0|             0|
|2025-08-03| analytics|       25.4613|   343.0|             0|
|2025-08-03|   compute|       12.6292|   136.0|             0|
|2025-08-03|     genai|       19.2865|   124.0|             0|
|2025-08-03|networking|         1.402|   114.0|             0|
|2025-08-03|   storage|          2.39|   109.0|             0|
+----------+----------

5.3 P1 y P2 con Spark

Elige como ejemplo el cliente que más gastó en total (org_53lc58dr) y responde las dos preguntas sobre el Parquet de Gold:

* P1, costos diarios entre el 01/08 y el 03/08: cada día muestra solo los servicios que el cliente usó ese día. El 01/08 no aparece analytics porque ese día no tuvo eventos de analytics. anomaly_events en 0: días sin gastos raros.
* P2, top 5 servicios de los últimos 14 días: los últimos 14 días son del 18/08 al 31/08, porque el dataset termina el 31/08. Arriba de todo quedan analytics (282 USD) y genai (271). Ese es el tipo de respuesta que FinOps espera: "¿en qué gasta más este cliente últimamente?".

Estos resultados son la referencia. Cuando hagamos las mismas preguntas en Cassandra, tienen que dar exactamente esto. Si coinciden, la carga está bien.

## 5.4 Resumen de Gold


In [170]:
ver_registro("5 gold")

,momento,etapa,tabla,leidas,escritas,kb,ruta,dias
8,2026-11-25 17:37:02,5 gold,org_daily_usage_by_service,43134,11050,702.5,/content/lake/gold/org_daily_usage_by_service,60.0


| Regla | Cómo se cumple |
|---|---|
| Mart `org_daily_usage_by_service` con grano diario por cliente y servicio, métricas y costos (consigna) | 5.1 |
| Grano explícito | Una fila por cliente, día y servicio; `assert` en 5.2 |
| Sale solo de Silver | Lee `silver/usage_events` |
| Totales Gold = Silver | `assert` en 5.2 |
| Correspondencia con MapReduce (diseño 9) | Cálculo de referencia con `map` + `reduceByKey`, igual fila por fila |
| Reejecución sin duplicados | Escritura `overwrite` por partición de día |

**Fuera de esta entrega:** `cost_anomaly_mart` (streaming, con su checkpoint), `revenue_by_org_month`, `tickets_by_org_date` y `genai_tokens_by_org_date`. Quedan en el backlog para el final.

# 6. Serving: Cassandra (AstraDB)

Reglas (diseño 7.1 y 7.7, clase 14):
- Una tabla por consulta (query-first). Cada consulta lee **una sola partición**.
- Sin joins: los datos que la consulta necesita se guardan juntos en la fila (desnormalización).
- Pasa a Cassandra solo lo que ya se validó en Gold (5.2).

Se conecta con el driver de Cassandra para Python: el token va en los Secretos de Colab y nunca en el código ni en git.

## 6.1 Modelo query-first

| Consulta | Tabla | Clave de partición | Clustering | Cómo se lee |
|---|---|---|---|---|
| **P1.** Costos y requests diarios por servicio de un cliente, en un rango de fechas | `org_daily_usage_by_service` | `org_id` | `usage_date DESC`, `service` | Igualdad en `org_id` y rango en `usage_date` |
| **P2.** Top N servicios por costo acumulado en los últimos 14 días, para un cliente | `org_top_services_14d` | `org_id` | `rank` | Igualdad en `org_id` y `LIMIT N` |

- **P1:** la partición es el cliente y adentro las filas quedan ordenadas por fecha (la más nueva primero). Un rango de fechas es una lectura seguida dentro de una partición.
- **P2:** Cassandra no puede sumar por servicio ni ordenar por una suma (no hay `GROUP BY` sobre columnas que no son de la clave). Por eso el ranking lo calcula Spark y Cassandra lo guarda ya ordenado: la consulta es "las primeras N filas de la partición". Cada cliente tiene siempre sus 6 servicios (con 0 si no usó alguno), así que al volver a cargar, cada `rank` se pisa y no quedan filas viejas.
- **Tamaño de partición:** un cliente suma como máximo 6 filas por día (una por servicio), unas 2.200 por año. Es chico y parejo entre clientes. Si en producción crece mucho, se agrega el mes a la clave de partición (bucket), como en el ejemplo de la clase.
- **Replicación y consistencia:** AstraDB guarda 3 copias de cada partición en distintas zonas y el driver usa `LOCAL_QUORUM` (2 de 3 confirman), así que si cae un nodo se sigue leyendo y escribiendo.

> **Decisión D19 · Claves de Cassandra**
>
> Dos tablas, una por consulta, ambas con el cliente como clave de partición: P1 con clustering por fecha descendente y servicio; P2 con clustering por posición en el ranking. El ranking de 14 días se calcula en Spark porque Cassandra no agrega entre filas por columnas que no son de la clave.

## 6.2 Conexión

Antes de correr esta celda:
1. **Token:** ícono de la llave a la izquierda → Agregar nuevo secreto → nombre `ASTRA_TOKEN`, valor `AstraCS:…` → activar "Acceso del notebook".
2. **Secure Connect Bundle:** la celda lo pide si no está subido. Es el `.zip` que se descarga de Astra (Connection details); no se descomprime y no va al repo.

Si no hay token, la sección se saltea sin cortar el resto del notebook.

In [171]:
from cassandra.cluster import Cluster, ExecutionProfile, EXEC_PROFILE_DEFAULT
from cassandra.auth import PlainTextAuthProvider
from cassandra.concurrent import execute_concurrent_with_args
from cassandra.query import dict_factory

KEYSPACE = "cloud_analytics"
CASSANDRA_OK = False

try:
    from google.colab import userdata, files
    token = userdata.get("ASTRA_TOKEN")

    # Secure Connect Bundle: si no está en /content, se pide
    bundles = sorted(Path("/content").glob("secure-connect*.zip"))
    if not bundles:
        print("Subí el Secure Connect Bundle (.zip)")
        subido = files.upload()
        bundles = [Path("/content") / list(subido)[0]]

    perfil = ExecutionProfile(row_factory=dict_factory, request_timeout=60)   # filas como diccionarios
    cluster = Cluster(cloud={"secure_connect_bundle": str(bundles[0])},
                      auth_provider=PlainTextAuthProvider("token", token),
                      execution_profiles={EXEC_PROFILE_DEFAULT: perfil},
                      protocol_version=4)
    session = cluster.connect()        # sin keyspace fijo: las tablas se nombran como keyspace.tabla
    CASSANDRA_OK = True
    print("Conectado a AstraDB · keyspace", KEYSPACE,
          "· versión", session.execute("SELECT release_version FROM system.local").one()["release_version"])
except Exception as e:
    print("Sin conexión a Cassandra; se saltea la sección 6.", type(e).__name__, e)

Conectado a AstraDB · keyspace cloud_analytics · versión 4.0.11.0-6819aa9c7f6c


## 6.3 Tablas

`CREATE TABLE IF NOT EXISTS`: si la tabla ya existe no hace nada, así que la celda se puede volver a correr. El mismo CQL está en `cql/serving.cql`.

In [172]:
CQL_TABLAS = [
"""
CREATE TABLE IF NOT EXISTS cloud_analytics.org_daily_usage_by_service (
    org_id            text,
    usage_date        date,
    service           text,
    org_name          text,
    daily_cost_usd    double,
    requests          double,
    cpu_hours         double,
    storage_gb_hours  double,
    genai_tokens      bigint,
    carbon_kg         double,
    events            int,
    anomaly_events    int,
    anomaly_cost_usd  double,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC)
""",
"""
CREATE TABLE IF NOT EXISTS cloud_analytics.org_top_services_14d (
    org_id        text,
    rank          int,
    service       text,
    cost_usd_14d  double,
    requests_14d  double,
    window_start  date,
    window_end    date,
    PRIMARY KEY ((org_id), rank)
) WITH CLUSTERING ORDER BY (rank ASC)
"""]

if CASSANDRA_OK:
    for cql in CQL_TABLAS:
        session.execute(cql)
    print([t["table_name"] for t in session.execute(
        "SELECT table_name FROM system_schema.tables WHERE keyspace_name = %s", [KEYSPACE])])

['org_daily_usage_by_service', 'org_top_services_14d']


## 6.4 Ranking de 14 días (para P2)

Se calcula en Spark sobre Gold. La ventana son los 14 días que terminan en el último día con datos (el dataset termina el 31/08/2025; en producción sería "ayer").

In [173]:
SERVICIOS = [r["service"] for r in gold_uso.select("service").distinct().collect()]
fin = gold_uso.agg(F.max("usage_date")).first()[0]
inicio = fin - timedelta(days=13)

# Todos los clientes con todos los servicios (0 si no usó alguno)
base = gold_uso.select("org_id").distinct().crossJoin(spark.createDataFrame([(s,) for s in SERVICIOS], ["service"]))

suma_14d = (gold_uso.filter(F.col("usage_date").between(inicio, fin))
            .groupBy("org_id", "service")
            .agg(F.sum("daily_cost_usd").alias("cost_usd_14d"), F.sum("requests").alias("requests_14d")))

top_14d = (base.join(suma_14d, ["org_id", "service"], "left").fillna(0)
    .withColumn("rank", F.row_number().over(Window.partitionBy("org_id").orderBy(F.desc("cost_usd_14d"), "service")))
    .withColumn("cost_usd_14d", F.round("cost_usd_14d", 4))
    .withColumn("window_start", F.lit(inicio)).withColumn("window_end", F.lit(fin)))

top_14d.write.mode("overwrite").parquet(str(GOLD / "org_top_services_14d"))
print(f"Ventana: {inicio} a {fin} | filas: {top_14d.count()}")
top_14d.filter(F.col("org_id") == ORG_EJEMPLO).orderBy("rank").show()

Ventana: 2025-08-18 a 2025-08-31 | filas: 480
+------------+----------+------------+------------+----+------------+----------+
|      org_id|   service|cost_usd_14d|requests_14d|rank|window_start|window_end|
+------------+----------+------------+------------+----+------------+----------+
|org_53lc58dr| analytics|    282.0746|      3671.0|   1|  2025-08-18|2025-08-31|
|org_53lc58dr|     genai|    271.2743|      2016.0|   2|  2025-08-18|2025-08-31|
|org_53lc58dr|   compute|     237.477|      2807.0|   3|  2025-08-18|2025-08-31|
|org_53lc58dr|  database|    182.4848|      3103.0|   4|  2025-08-18|2025-08-31|
|org_53lc58dr|   storage|      36.266|      1556.0|   5|  2025-08-18|2025-08-31|
|org_53lc58dr|networking|     15.4267|      1261.0|   6|  2025-08-18|2025-08-31|
+------------+----------+------------+------------+----+------------+----------+



## 6.5 Carga desde Spark

Spark entrega las filas de Gold y el driver las inserta en paralelo (`execute_concurrent_with_args`, 50 a la vez) con una sentencia preparada.

En Cassandra un `INSERT` con la misma clave primaria **pisa** la fila (upsert): volver a cargar no duplica.

In [174]:
def cargar(df, tabla, columnas):
    # Inserta un DataFrame de Spark en una tabla de Cassandra
    sentencia = session.prepare(
          f"INSERT INTO {KEYSPACE}.{tabla} ({', '.join(columnas)}) VALUES ({', '.join('?' * len(columnas))})")
    filas = [tuple(r[c] for c in columnas) for r in df.select(columnas).toLocalIterator()]
    resultados = execute_concurrent_with_args(session, sentencia, filas, concurrency=50, raise_on_first_error=True)
    return sum(1 for ok, _ in resultados if ok)


COLS_USO = ["org_id", "usage_date", "service", "org_name", "daily_cost_usd", "requests", "cpu_hours",
            "storage_gb_hours", "genai_tokens", "carbon_kg", "events", "anomaly_events", "anomaly_cost_usd"]
COLS_TOP = ["org_id", "rank", "service", "cost_usd_14d", "requests_14d", "window_start", "window_end"]

if CASSANDRA_OK:
    gold_cassandra = (gold_uso
        .withColumn("genai_tokens", F.col("genai_tokens").cast("long"))
        .withColumn("events", F.col("events").cast("int"))
        .withColumn("anomaly_events", F.col("anomaly_events").cast("int")))

    t0 = datetime.now()
    n_uso = cargar(gold_cassandra, "org_daily_usage_by_service", COLS_USO)
    n_top = cargar(top_14d.withColumn("rank", F.col("rank").cast("int")), "org_top_services_14d", COLS_TOP)
    segundos = (datetime.now() - t0).seconds

    anotar("6 serving", "org_daily_usage_by_service", leidas=gold_uso.count(), escritas=n_uso, segundos=segundos)
    anotar("6 serving", "org_top_services_14d", leidas=top_14d.count(), escritas=n_top, segundos=segundos)

## 6.6 Control: lo que hay en Cassandra coincide con Gold

Se cuenta partición por partición (una consulta por cliente), que es la forma barata de contar en Cassandra: un `COUNT(*)` sobre toda la tabla recorre el clúster entero.

In [175]:
if CASSANDRA_OK:
    clientes_gold = [r["org_id"] for r in gold_uso.select("org_id").distinct().collect()]
    contar = session.prepare("SELECT COUNT(*) AS n FROM cloud_analytics.org_daily_usage_by_service WHERE org_id = ?")
    en_cassandra = sum(session.execute(contar, [o]).one()["n"] for o in clientes_gold)

    print(f"Gold: {gold_uso.count()} filas | Cassandra: {en_cassandra} filas")
    assert en_cassandra == gold_uso.count(), "Cassandra no tiene las mismas filas que Gold"

Gold: 11050 filas | Cassandra: 11050 filas


## 6.7 Consultas

Las dos consultas, con el CQL tal como se ejecuta. El resultado se compara con lo calculado en Spark sobre Gold (5.3): tiene que ser igual.

Para la evidencia, el mismo CQL se puede correr en la consola CQL de Astra (pestaña CQL Console) y sacar la captura.

In [176]:
from datetime import date

DESDE, HASTA = date(2025, 8, 1), date(2025, 8, 3)

CQL_P1 = (f"SELECT usage_date, service, daily_cost_usd, requests, anomaly_events "
          f"FROM cloud_analytics.org_daily_usage_by_service "
          f"WHERE org_id = '{ORG_EJEMPLO}' AND usage_date >= '{DESDE}' AND usage_date <= '{HASTA}'")

if CASSANDRA_OK:
    print(CQL_P1, "\n")
    p1 = pd.DataFrame(session.execute(CQL_P1))
    display(p1)

    # Mismo resultado que Spark sobre Gold
    p1_spark = (gold_uso.filter((F.col("org_id") == ORG_EJEMPLO) & F.col("usage_date").between(DESDE, HASTA))
                .select("usage_date", "service", "daily_cost_usd").toPandas())
    assert len(p1) == len(p1_spark), "P1: distinta cantidad de filas que en Gold"
    assert abs(p1["daily_cost_usd"].sum() - p1_spark["daily_cost_usd"].sum()) < 0.01, "P1: distinto costo que en Gold"

SELECT usage_date, service, daily_cost_usd, requests, anomaly_events FROM cloud_analytics.org_daily_usage_by_service WHERE org_id = 'org_53lc58dr' AND usage_date >= '2025-08-01' AND usage_date <= '2025-08-03' 



,usage_date,service,daily_cost_usd,requests,anomaly_events
0,2025-08-03,analytics,25.4613,343.0,0
1,2025-08-03,compute,12.6292,136.0,0
2,2025-08-03,genai,19.2865,124.0,0
3,2025-08-03,networking,1.4020,114.0,0
4,2025-08-03,storage,2.3900,109.0,0
5,2025-08-02,analytics,32.7553,478.0,0
6,2025-08-02,database,1.5619,0.0,0
7,2025-08-02,networking,2.2607,237.0,0
8,2025-08-02,storage,4.8398,255.0,0
9,2025-08-01,database,1.5839,0.0,0


In [177]:
TOP_N = 5

CQL_P2 = (f"SELECT rank, service, cost_usd_14d, requests_14d, window_start, window_end "
          f"FROM cloud_analytics.org_top_services_14d "
          f"WHERE org_id = '{ORG_EJEMPLO}' LIMIT {TOP_N}")

if CASSANDRA_OK:
    print(CQL_P2, "\n")
    p2 = pd.DataFrame(session.execute(CQL_P2))
    display(p2)

    p2_spark = top_14d.filter((F.col("org_id") == ORG_EJEMPLO) & (F.col("rank") <= TOP_N)).orderBy("rank").toPandas()
    assert list(p2["service"]) == list(p2_spark["service"]), "P2: distinto ranking que en Spark"

SELECT rank, service, cost_usd_14d, requests_14d, window_start, window_end FROM cloud_analytics.org_top_services_14d WHERE org_id = 'org_53lc58dr' LIMIT 5 



,rank,service,cost_usd_14d,requests_14d,window_start,window_end
0,1,analytics,282.0746,3671.0,2025-08-18,2025-08-31
1,2,genai,271.2743,2016.0,2025-08-18,2025-08-31
2,3,compute,237.4770,2807.0,2025-08-18,2025-08-31
3,4,database,182.4848,3103.0,2025-08-18,2025-08-31
4,5,storage,36.2660,1556.0,2025-08-18,2025-08-31


In [178]:
# CQL listo para pegar en la consola CQL de Astra (captura para la evidencia)
print(CQL_P1 + ";\n")
print(CQL_P2 + ";")

SELECT usage_date, service, daily_cost_usd, requests, anomaly_events FROM cloud_analytics.org_daily_usage_by_service WHERE org_id = 'org_53lc58dr' AND usage_date >= '2025-08-01' AND usage_date <= '2025-08-03';

SELECT rank, service, cost_usd_14d, requests_14d, window_start, window_end FROM cloud_analytics.org_top_services_14d WHERE org_id = 'org_53lc58dr' LIMIT 5;


## 6.8 Resumen de Serving

| Regla | Cómo se cumple |
|---|---|
| Keyspace y tabla query-first para el mart (consigna) | Keyspace `cloud_analytics`; dos tablas, una por consulta (6.1, 6.3) |
| Carga desde Spark (consigna) | 6.5, con conteo partición por partición igual a Gold (6.6) |
| Dos consultas con CQL y resultados (consigna) | P1 y P2 en 6.7; CQL también en `cql/serving.cql` |
| Cada consulta lee una sola partición (clase 14) | Ambas filtran por `org_id` |
| Pasa a Cassandra solo lo validado | Carga después de los `assert` de 5.2 |
| Recargar no duplica | `INSERT` = upsert por clave primaria |
| Token fuera del código y de git | Secretos de Colab |

# 7. Pruebas e idempotencia

Criterio de aceptación de la consigna: **"La re-ejecución no duplica registros."** Se prueba de tres maneras:

1. **Reejecutar los streams** con los mismos checkpoints: no tienen que leer nada de nuevo.
2. **La fuente reenvía un archivo** (un error común: el mismo archivo llega dos veces con otro nombre): la deduplicación por `event_id` no tiene que dejar entrar repetidos.
3. **Reejecutar los procesos batch** (maestros, Gold y carga en Cassandra): `overwrite` y upsert reemplazan, no agregan.

Se cuentan las filas de cada tabla antes y después.

## 7.1 Conteos antes

In [179]:
def contar_todo():
    # Filas de cada tabla del lake (y de Cassandra si hay conexión)
    tablas = {
        "bronze/usage_events":          BRONZE / "usage_events",
        "bronze/customers_orgs":        BRONZE / "customers_orgs",
        "bronze/billing_monthly":       BRONZE / "billing_monthly",
        "silver/usage_events":          SILVER / "usage_events",
        "silver/quarantine/usage_events": QUARANTINE / "usage_events",
        "gold/org_daily_usage_by_service": GOLD / "org_daily_usage_by_service",
    }
    conteo = {nombre: spark.read.parquet(str(ruta)).count() for nombre, ruta in tablas.items()}
    conteo["event_id distintos en silver"] = spark.read.parquet(str(SILVER / "usage_events")).select("event_id").distinct().count()
    if CASSANDRA_OK:
        contar = session.prepare(f"SELECT COUNT(*) AS n FROM {KEYSPACE}.org_daily_usage_by_service WHERE org_id = ?")
        orgs = [r["org_id"] for r in spark.read.parquet(str(GOLD / "org_daily_usage_by_service")).select("org_id").distinct().collect()]
        conteo["cassandra/org_daily_usage_by_service"] = sum(session.execute(contar, [o]).one()["n"] for o in orgs)
    return conteo

antes = contar_todo()
pd.DataFrame({"antes": antes})

,antes
bronze/usage_events,43171
bronze/customers_orgs,80
bronze/billing_monthly,240
silver/usage_events,43134
silver/quarantine/usage_events,37
gold/org_daily_usage_by_service,11050
event_id distintos en silver,43134
cassandra/org_daily_usage_by_service,11050


## 7.2 Reejecutar los streams

Se vuelven a lanzar las tres consultas de streaming con sus mismos checkpoints. El checkpoint tiene anotados todos los archivos procesados, así que no tiene que entrar ninguna fila.

In [180]:
def relanzar(df, destino, checkpoint, nombre, particion=None):
    # Misma consulta, mismo checkpoint; devuelve cuántas filas leyó
    escritor = (df.writeStream.format("parquet")
                .option("path", str(destino)).option("checkpointLocation", str(checkpoint))
                .trigger(availableNow=True).queryName(nombre))
    if particion:
        escritor = escritor.partitionBy(particion)
    q = escritor.start()
    q.awaitTermination()
    return sum(p["numInputRows"] for p in q.recentProgress)

leidas = {
    "bronze_usage_events": relanzar(bronze_eventos, BRONZE / "usage_events",
                                    CHECKPOINTS / "bronze_usage_events", "bronze_usage_events", "event_date"),
    "silver_usage_events": relanzar(evaluados.filter(F.col("regla").isNull()).select(COLUMNAS_SILVER),
                                    SILVER / "usage_events", CHECKPOINTS / "silver_usage_events",
                                    "silver_usage_events", "event_date"),
}
print("Filas leídas al reejecutar:", leidas)
assert sum(leidas.values()) == 0, "El checkpoint no evitó volver a leer archivos"

Filas leídas al reejecutar: {'bronze_usage_events': 0, 'silver_usage_events': 0}


## 7.3 La fuente reenvía un archivo

Se copia uno de los últimos archivos de eventos con otro nombre, como si el sistema de medición lo mandara dos veces. Para Spark es un archivo nuevo (el checkpoint no lo conoce), así que lo lee; la deduplicación por `event_id` dentro del watermark (D16) tiene que descartar todos sus eventos.

In [181]:
# Entre los últimos 50 archivos, el que más eventos tiene
recientes = sorted((LANDING / "usage_events_stream").glob("events_*.jsonl"))[-50:]
original = max(recientes, key=lambda f: len(f.read_text(encoding="utf-8").splitlines()))
reenvio = original.with_name(original.stem + "_reenvio.jsonl")
shutil.copy2(original, reenvio)
os.utime(reenvio, None)          # llega "ahora": es el archivo más nuevo

eventos_reenviados = len(reenvio.read_text(encoding="utf-8").splitlines())
leidas_reenvio = relanzar(bronze_eventos, BRONZE / "usage_events",
                          CHECKPOINTS / "bronze_usage_events", "bronze_usage_events", "event_date")

bronze_despues = spark.read.parquet(str(BRONZE / "usage_events"))
print(f"Archivo reenviado: {reenvio.name} ({eventos_reenviados} eventos)")
print(f"Filas leídas por el stream: {leidas_reenvio}")
print(f"Bronze antes: {antes['bronze/usage_events']} | después: {bronze_despues.count()}")
assert bronze_despues.count() == antes["bronze/usage_events"], "Entraron eventos repetidos a Bronze"
assert bronze_despues.select("event_id").distinct().count() == bronze_despues.count()

reenvio.unlink()                 # se retira el archivo de prueba de Landing

Archivo reenviado: events_20250831T2130_reenvio.jsonl (13 eventos)
Filas leídas por el stream: 13
Bronze antes: 43171 | después: 43171


## 7.4 Reejecutar los procesos batch

Maestros a Bronze, Gold y la carga en Cassandra, otra vez.

In [182]:
# Maestros a Bronze, otra vez
for tabla in ["customers_orgs", "billing_monthly"]:
    bronze_maestro(tabla)

# Gold: se recalcula desde Silver y se reescriben los días (overwrite dinámico)
gold_calculado.write.mode("overwrite").partitionBy("usage_date").parquet(str(GOLD / "org_daily_usage_by_service"))
gold_uso = spark.read.parquet(str(GOLD / "org_daily_usage_by_service"))   # se vuelve a leer: los archivos cambiaron

# Cassandra: se vuelve a cargar lo mismo (upsert)
if CASSANDRA_OK:
    cargar(gold_uso.withColumn("genai_tokens", F.col("genai_tokens").cast("long"))
                   .withColumn("events", F.col("events").cast("int"))
                   .withColumn("anomaly_events", F.col("anomaly_events").cast("int")),
           "org_daily_usage_by_service", COLS_USO)

## 7.5 Conteos después

In [183]:
despues = contar_todo()
idempotencia = pd.DataFrame({"antes": antes, "después": despues})
idempotencia["diferencia"] = idempotencia["después"] - idempotencia["antes"]

assert (idempotencia["diferencia"] == 0).all(), "La reejecución cambió la cantidad de filas"
anotar("7 pruebas", "idempotencia", tablas=len(idempotencia), diferencias=int(idempotencia["diferencia"].abs().sum()))
idempotencia

,antes,después,diferencia
bronze/usage_events,43171,43171,0
bronze/customers_orgs,80,80,0
bronze/billing_monthly,240,240,0
silver/usage_events,43134,43134,0
silver/quarantine/usage_events,37,37,0
gold/org_daily_usage_by_service,11050,11050,0
event_id distintos en silver,43134,43134,0
cassandra/org_daily_usage_by_service,11050,11050,0


## 7.6 Pruebas automáticas

Resumen de los controles (`assert`) que corren en cada ejecución. Si alguno falla, el notebook se detiene en esa celda con el mensaje del problema.

| Prueba | Dónde | Qué verifica |
|---|---|---|
| Claves únicas en maestros | 3.2 | Una fila por clave en Bronze |
| Completitud Landing → Bronze | 3.8 | Landing = Bronze + tardíos en cuarentena |
| Sin duplicados en Bronze | 3.8 | Un `event_id` por fila |
| Sin pérdidas Bronze → Silver | 4.7 | Bronze = Silver + cuarentena |
| Sin duplicados en Silver | 4.7 | Un `event_id` por fila |
| Grano de Gold | 5.2 | Una fila por cliente, día y servicio |
| Totales Gold = Silver | 5.2 | Costo, requests y eventos |
| Gold = MapReduce de referencia | 5.2 | Igual fila por fila |
| Cassandra = Gold | 6.6 | Misma cantidad de filas |
| Consultas = Spark | 6.7 | P1 y P2 devuelven lo mismo que Gold |
| Streams no releen | 7.2 | 0 filas al reejecutar con checkpoint |
| Archivo reenviado | 7.3 | 0 eventos nuevos en Bronze |
| Reejecución sin duplicados | 7.5 | Mismos conteos antes y después |

# 8. Evidencias

Se guardan en `evidence/` del repo: registro de ejecución, inventario del lake (rutas y tamaños), registro de llegadas, muestras de cuarentena, conteos de idempotencia y resultados de las consultas.

Como el lake vive en Colab, al final se descarga un `.zip` con la carpeta `evidence/` para subirla a GitHub.

In [184]:
# Registro de ejecución y de llegadas
ver_registro().to_csv(EVIDENCE / "registro_ejecucion.csv", index=False)
registro_llegadas.to_csv(EVIDENCE / "registro_llegadas_landing.csv", index=False)

# Inventario del lake: cada tabla con su ruta, cantidad de archivos y tamaño
inventario = []
for zona in [BRONZE, SILVER, QUARANTINE, GOLD]:
    for tabla in sorted(p for p in zona.iterdir() if p.is_dir() and p != QUARANTINE):
        archivos = list(tabla.rglob("*.parquet"))
        inventario.append({"zona": zona.name, "tabla": tabla.name, "ruta": str(tabla),
                           "archivos_parquet": len(archivos),
                           "kb": round(sum(f.stat().st_size for f in archivos) / 1024, 1)})
pd.DataFrame(inventario).to_csv(EVIDENCE / "inventario_lake.csv", index=False)

# Muestras de cuarentena (hasta 20 por regla)
for nombre in ["usage_events", "usage_events_late"]:
    (spark.read.parquet(str(QUARANTINE / nombre))
     .withColumn("n", F.row_number().over(Window.partitionBy("regla").orderBy("event_id")))
     .filter("n <= 20").drop("n")
     .toPandas().to_csv(EVIDENCE / f"muestra_cuarentena_{nombre}.csv", index=False))

# Idempotencia, umbrales y consultas
idempotencia.to_csv(EVIDENCE / "idempotencia_conteos.csv")
spark.read.parquet(str(SILVER / "ref_cost_thresholds")).toPandas().to_csv(EVIDENCE / "umbrales_anomalia.csv", index=False)
if CASSANDRA_OK:
    p1.to_csv(EVIDENCE / "cassandra_p1.csv", index=False)
    p2.to_csv(EVIDENCE / "cassandra_p2.csv", index=False)

pd.DataFrame(inventario)

,zona,tabla,ruta,archivos_parquet,kb
0,bronze,billing_monthly,/content/lake/bronze/billing_monthly,3,21.5
1,bronze,customers_orgs,/content/lake/bronze/customers_orgs,1,7.2
2,bronze,resources,/content/lake/bronze/resources,1,12.1
3,bronze,usage_events,/content/lake/bronze/usage_events,541,5775.0
4,bronze,users,/content/lake/bronze/users,1,23.7
5,silver,customers_orgs,/content/lake/silver/customers_orgs,1,7.2
6,silver,ref_cost_thresholds,/content/lake/silver/ref_cost_thresholds,1,2.2
7,silver,resources,/content/lake/silver/resources,1,13.5
8,silver,usage_events,/content/lake/silver/usage_events,151,4473.8
9,quarantine,customers_orgs,/content/lake/silver/quarantine/customers_orgs,1,1.9


In [185]:
# Descargar evidence/ como .zip (solo en Colab)
archivo_zip = shutil.make_archive("/content/evidence", "zip", EVIDENCE)
print("Evidencias:", sorted(f.name for f in EVIDENCE.iterdir()))
try:
    from google.colab import files
    files.download(archivo_zip)
except ImportError:
    pass

Evidencias: ['cassandra_p1.csv', 'cassandra_p2.csv', 'idempotencia_conteos.csv', 'inventario_lake.csv', 'muestra_cuarentena_usage_events.csv', 'muestra_cuarentena_usage_events_late.csv', 'registro_ejecucion.csv', 'registro_llegadas_landing.csv', 'umbrales_anomalia.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>